# Custom Chatbot Project

### Dataset Choice and Scenario

This project explores the implementation of a retrieval-augmented generation (RAG) system by designing a chatbot that queries and responds based on external data it was not trained on orginally.

For my project, I created a custom dataset called matcha_knowledge.csv, around one of my favorite things: matcha. 🍵 This dataset includes over 20 curated entries on topics like matcha grades, brewing methods (usucha vs. koicha), traditional tools, health benefits, and famous growing regions in Japan.


![Matcha_Gif](https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExdDRiMjg4azFsbXR0MjhiM2t4c2s4OGVwbXcyYjc4MndmZWVpMGp0OCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/v0VoVGS0KrJZf1Yz6L/giphy.gif)

Unlike base models that often verbose or can hallucinate, my custom bot delivers concise, factual answers grounded in context. That’s important when dealing with specialized cultural topics like matcha, where clarity matters more than flair.




In [56]:
!pip install openai==0.28 tiktoken --quiet
print("Packages installed")

Packages installed


In [57]:
import pandas as pd
import numpy as np
import openai
import os
import tiktoken
from scipy.spatial.distance import cosine
from openai.embeddings_utils import distances_from_embeddings

In [ ]:
openai.api_base = "https://openai.vocareum.com/v1"
openai.api_key = "YOUR_API_KEY"
print("OpenAI API key set.")

OpenAI API key set.


## Data Wrangling

In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [65]:
#Load the matcha dataset
df = pd.read_csv('/content/matcha_knowledge.csv')

df.describe()

,topic,description
count,20,20
unique,20,20
top,Matcha Grades,There are ceremonial and culinary grades of ma...
freq,1,1


In [66]:
#Create a text column combining topic and description
df['text'] = df['topic'] + ': ' + df['description']

#Display the first few entries
df[['text']].head()

,text
0,Matcha Grades: There are ceremonial and culina...
1,Usucha Preparation: Usucha is thin matcha. Use...
2,Koicha Preparation: Koicha is thick matcha. Us...
3,Matcha Health Benefits: Matcha is rich in cate...
4,Matcha Tools: Traditional tools include the Ch...


In [67]:
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"

embeddings = []
for index, row in df.iterrows():
  response = openai.Embedding.create(
      input=row["text"],
      engine=EMBEDDING_MODEL_NAME
  )
  embeddings.extend([data["embedding"] for data in response["data"]])
df["embeddings"] = embeddings

In [68]:
df[["text", "embeddings"]].to_csv("matcha_knowledge_embeddings.csv", index=False)

In [69]:
df = pd.read_csv('/content/matcha_knowledge_embeddings.csv')
df["embeddings"] = df["embeddings"].apply(eval).apply(np.array)

df.head()

,text,embeddings
0,Matcha Grades: There are ceremonial and culina...,"[0.01780729554593563, 0.009216392412781715, 0...."
1,Usucha Preparation: Usucha is thin matcha. Use...,"[-0.0012188827386125922, 0.01881968043744564, ..."
2,Koicha Preparation: Koicha is thick matcha. Us...,"[0.005617733113467693, 0.0047905626706779, 0.0..."
3,Matcha Health Benefits: Matcha is rich in cate...,"[-0.002339455531910062, -0.0014507108135148883..."
4,Matcha Tools: Traditional tools include the Ch...,"[-0.010844801552593708, 0.001961293863132596, ..."


## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

In [70]:
def build_simple_prompt(question: str):
    return [
        {"role": "user", "content": question}
    ]

In [71]:
def get_embedding(text: str, model="text-embedding-ada-002"):
    result = openai.Embedding.create(input=text, engine=model)
    return result["data"][0]["embedding"]

def build_custom_context(question: str, df: pd.DataFrame, n: int = 5):
    question_embedding = get_embedding(question)
    df["distance"] = df["embeddings"].apply(lambda x: cosine(x, question_embedding))
    top_contexts = df.sort_values("distance").head(n)
    return top_contexts["text"].tolist()


In [72]:
def build_custom_prompt(question: str, df: pd.DataFrame):
    context = "\n\n".join(build_custom_context(question, df))
    return [
        {"role": "system", "content": f"""Answer the question using the context below.
If you don't know the answer from the context, say "I don't know."

Context:
{context}
"""},
        {"role": "user", "content": question}
    ]

In [77]:
MAX_TOKENS=150

def get_response(prompt, model="gpt-3.5-turbo"):
    response = openai.ChatCompletion.create(
        model=model,
        messages=prompt,
        max_tokens=MAX_TOKENS
    )
    return response["choices"][0]["message"]["content"].strip()

In [74]:
def compare_responses(question: str, df: pd.DataFrame):
    print(f"Question: {question}\n")

    simple_prompt = build_simple_prompt(question)
    custom_prompt = build_custom_prompt(question, df)

    base_response = get_response(simple_prompt)
    custom_response = get_response(custom_prompt)

    print("Base Model Response:")
    print(base_response)
    print("\n Custom RAG-Based Response:")
    print(custom_response)

## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [75]:
compare_responses("What is the best temperature for matcha?", df)

Question: What is the best temperature for matcha?

Base Model Response:
The optimal temperature for preparing matcha is around 175-180°F (80-82°C). Water that is too hot can scorch the delicate matcha powder and result in a bitter taste, while water that is too cool may not fully extract the flavors of the matcha. It is recommended to use a thermometer to ensure the water is at the correct temperature before whisking it with the matcha powder.

 Custom RAG-Based Response:
The best temperature for matcha is between 70-80°C (158-176°F).


### Question 2

In [76]:
compare_responses("What tools are essential for preparing traditional matcha and how are they used?", df)

Question: What tools are essential for preparing traditional matcha and how are they used?

Base Model Response:
1. Chashaku (bamboo scoop): The chashaku is used to scoop the matcha powder from the container into the tea bowl. It is essential for measuring the correct amount of matcha for one serving.

2. Chasen (bamboo whisk): The chasen is used to whisk the matcha powder and hot water together to create a frothy and smooth matcha tea. The whisk is made of bamboo and has fine prongs that help to mix the matcha thoroughly.

3. Chawan (tea bowl): The chawan is a wide, shallow bowl that is used to prepare and drink matcha. It is essential for whisking the matcha and hot water together, as the wide shape allows for easy

 Custom RAG-Based Response:
The traditional tools essential for preparing matcha are the Chasen (bamboo whisk), Chawan (bowl), Chashaku (scoop), and Natsume (container). The Chasen is used to briskly whisk the matcha in a 'W' or 'M' motion until fine foam forms. The Chawa